# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Bài làm. Mã nằm trong thư mục `lab10/` (notebook này gọi lại các hàm đó theo từng bước của `GUIDE.md`).
Muốn chạy hết một lần trên Kaggle thì dùng `kaggle_lab10.ipynb` (xem `KAGGLE.md`).

## 0. Môi trường (Colab T4 hoặc Kaggle T4)

In [ ]:
# Chạy được trên cả Kaggle lẫn Colab. Trên Colab: Runtime → Change runtime type → T4 GPU.
import os, subprocess, sys
REPO_URL = "https://github.com/tubepvhat1604-bot/K4-Track-4-Day10-VisionLanguageModel.git"
BRANCH = "main"
if not os.path.exists("scripts/score.py"):                 # chưa ở trong repo → clone về
    if not os.path.exists("lab"):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, "lab"], check=True)
    os.chdir("lab")
print("thư mục làm việc:", os.getcwd())
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

In [ ]:
!python -m pytest -q

## 1. Dữ liệu: tải và chia split

In [ ]:
# Tìm dataset: Kaggle (kagglehub, không cần khoá) → /kaggle/input → Kaggle CLI (Colab Secrets).
import glob, os, subprocess

def find_dataset():
    try:
        import kagglehub
        p = kagglehub.dataset_download("lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes")
        if glob.glob(os.path.join(p, "**", "labels-YOLO"), recursive=True):
            return p
    except Exception as e:
        print("kagglehub không dùng được:", e)
    hits = glob.glob("/kaggle/input/**/labels-YOLO", recursive=True)
    return os.path.dirname(hits[0]) if hits else None

if not os.path.exists("data/splits/meta.json"):
    src = find_dataset()
    if src:
        subprocess.run(["python", "scripts/prepare_data.py", "--source", src], check=True)
    else:   # Colab: KAGGLE_USERNAME / KAGGLE_KEY trong Colab Secrets, KHÔNG viết khoá vào notebook
        from google.colab import userdata
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
        subprocess.run(["python", "scripts/prepare_data.py", "--kaggle"], check=True)
!cat data/splits/meta.json | head -5

In [ ]:
import json
from lab10.common import split_stats
print(json.dumps(split_stats(), indent=1))   # số ảnh/box từng split, session_overlap phải toàn 0

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
from PIL import Image, ImageDraw
from lab10.common import load_split, objects_to_answer, box_to_pixels, yolo_to_box
train = load_split("train")
e = next(r for r in train if len(r["objects"]) >= 2)
ans = objects_to_answer(e["objects"])
print(ans)
# Kiểm tra: JSON 0–1000 → pixel → vẽ (đỏ) phải trùng box YOLO gốc (xanh)
im = Image.open("data/" + e["image_path"]).convert("RGB"); d = ImageDraw.Draw(im)
for o in e["objects"]:
    xc, yc, w, h = o["yolo"]
    d.rectangle([(xc - w/2) * 640, (yc - h/2) * 360, (xc + w/2) * 640, (yc + h/2) * 360], outline="lime", width=5)
for b in json.loads(ans):
    d.rectangle(box_to_pixels(b["bbox_2d"]), outline="red", width=2)
im

## 3. E0: zero-shot

In [ ]:
!python -m lab10.run stats e0     # nạp model 4-bit, suy luận 311 ảnh test, chấm bằng scripts/score.py

In [ ]:
from scripts.labcommon import read_jsonl
for r in read_jsonl("outputs/preds/e0.jsonl")[:5]:
    print(repr(r["raw"][:300]), "\n")

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
print(json.dumps(json.load(open("outputs/stats.json"))["example"], indent=1, ensure_ascii=False))
# image_tokens phải là 220; decoded_loss_part phải đúng là JSON đáp án + <|im_end|>

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
print(json.dumps(json.load(open("outputs/stats.json"))["param_counts"], indent=1))
# Số module LoRA (252 / 252 / 348) và số tham số huấn luyện được in ra khi chạy bước 6.

## 6. Vòng huấn luyện

In [ ]:
!python -m lab10.run e1 e2 e3     # mỗi setup: train 300 bước → lưu adapter (+ projector) → suy luận → chấm

## 7. Suy luận và chấm E1–E3

In [ ]:
for e in ["e0", "e1", "e2", "e3"]:
    d = json.load(open(f"outputs/evidence/{e}.json"))
    print(e, round(d["micro"]["f1"], 4), [round(x, 4) for x in d["f1_ci95"]], d["parse_valid_rate"], d.get("meta"))

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
!python -m lab10.run e4 extras e5

In [ ]:
!mkdir -p report/evidence && cp outputs/evidence/*.json report/evidence/
!python scripts/check_submission.py